In [1]:
from datetime import date


def days_360(d1, d2):
    """Days between two dates using the 30/360 convention."""
    return 360 * (d2.year - d1.year) + 30 * (d2.month - d1.month) + (min(d2.day, 30) - min(d1.day, 30))


def check_inputs(face, coupon_rate, ytm, n_remaining, freq):
    if face <= 0:
        raise ValueError("face must be positive")
    if coupon_rate < 0:
        raise ValueError("coupon_rate cannot be negative")
    if ytm < 0:
        raise ValueError("ytm cannot be negative")
    if n_remaining < 1:
        raise ValueError("n_remaining must be at least 1")
    if freq not in (1, 2, 4, 12):
        raise ValueError("freq must be 1, 2, 4 or 12")


def accrued_interest(face, coupon_rate, last_coupon, settlement, freq=2):
    if settlement < last_coupon:
        raise ValueError("settlement cannot be before the last coupon date")
    period_days = 360 / freq
    days = days_360(last_coupon, settlement)
    return face * coupon_rate / freq * days / period_days


def clean_dirty(face, coupon_rate, ytm, n_remaining, last_coupon, settlement, freq=2):
    """
    n_remaining = number of coupons still to be paid after settlement
    (including the next one). Returns (clean, dirty, accrued).
    """
    check_inputs(face, coupon_rate, ytm, n_remaining, freq)
    period_days = 360 / freq
    days = days_360(last_coupon, settlement)
    if days > period_days:
        raise ValueError("settlement is more than one coupon period after last_coupon")
    w = 1 - days / period_days            # fraction of period until next coupon
    c = face * coupon_rate / freq
    y = ytm / freq

    dirty = 0
    for k in range(n_remaining):
        cash_flow = c
        if k == n_remaining - 1:
            cash_flow = c + face
        dirty = dirty + cash_flow / (1 + y) ** (k + w)

    accrued = accrued_interest(face, coupon_rate, last_coupon, settlement, freq)
    clean = dirty - accrued
    return clean, dirty, accrued

In [2]:
last = date(2026, 1, 15)
settle = date(2026, 4, 15)

clean, dirty, accrued = clean_dirty(1000, 0.08, 0.10, 9, last, settle)
print("Clean:", clean)
print("Dirty:", dirty)
print("Accrued:", accrued)

Clean: 931.8615778324145
Dirty: 951.8615778324145
Accrued: 20.0


In [3]:
# 1. Accrued interest matches the hand calculation
assert abs(accrued_interest(1000, 0.08, date(2026, 1, 15), date(2026, 4, 15)) - 20) < 1e-9

# 2. Clean + accrued = dirty
clean, dirty, accrued = clean_dirty(1000, 0.08, 0.10, 9, date(2026, 1, 15), date(2026, 4, 15))
assert abs(clean + accrued - dirty) < 1e-9

# 3. Settlement exactly on a coupon date: no accrued, price equals Day 1 result (10 periods = 5 years)
clean, dirty, accrued = clean_dirty(1000, 0.08, 0.10, 10, date(2026, 1, 15), date(2026, 1, 15))
assert accrued == 0
assert abs(clean - 922.7826507) < 1e-6

# 4. Zero-coupon bond: no accrued interest at any date
clean, dirty, accrued = clean_dirty(1000, 0.0, 0.10, 10, date(2026, 1, 15), date(2026, 4, 15))
assert accrued == 0

# 5. Zero yield, on a coupon date: price = face + all coupons = 1400
clean, dirty, accrued = clean_dirty(1000, 0.08, 0.0, 10, date(2026, 1, 15), date(2026, 1, 15))
assert abs(clean - 1400) < 1e-9

# 6. Bond maturing in under one period (1 coupon left): still works, price is positive
clean, dirty, accrued = clean_dirty(1000, 0.08, 0.10, 1, date(2026, 1, 15), date(2026, 4, 15))
assert dirty > 0 and accrued == 20

# 7. Negative yield raises a clear error
try:
    clean_dirty(1000, 0.08, -0.01, 10, date(2026, 1, 15), date(2026, 1, 15))
    assert False, "should have raised an error"
except ValueError:
    pass

# 8. Settlement before the last coupon date raises an error
try:
    accrued_interest(1000, 0.08, date(2026, 4, 15), date(2026, 1, 15))
    assert False, "should have raised an error"
except ValueError:
    pass

print("All 8 tests passed")

All 8 tests passed
